# core

> the vault: one SQLite file holding everything you have read, and the retrieval over it

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A `Vault` is a [litesearch](https://github.com/Karthik777/litesearch) document tree — docs → nodes →
chunks, FTS5 and a usearch HNSW index — with an entity graph over it, in one SQLite file. Everything
you read goes in under a `kind`, and one query crosses all of them. Nothing here reimplements
litesearch: the vault holds an encoder and a store name, and hands both to it.

In [ ]:
#| export
import json, re, uuid, warnings
from functools import partial
import numpy as np
from fastcore.all import AttrDict, L, Path, ifnone, patch
from litesearch import (database, dir2files, doc_encoder, query_encoder, hash_embed, static_embedder,
        pdf_parse, build_graph, resolve_entities, DOC_EXTS)

In [ ]:
#| export
KINDS = ('web', 'pdf', 'arxiv', 'youtube', 'file', 'code', 'data', 'note')
_window, DFLT_ENC = re.compile(r'^Pages \d+(?:–\d+)?: '), 'minishlab/potion-multilingual-128M'

def tidy_bc(bc:str) -> str:
    "Drop `build_tree`'s `Pages n–m:` window placeholders from a breadcrumb: real nodes, noise in a citation."
    return ' › '.join(dict.fromkeys(p for p in map(str.strip, (bc or '').split('›')) if p and not _window.match(p)))

def kinds(kind) -> L:
    "A kind filter as a list — `'note'`, `'note,web'` and `['note','web']` all work."
    return L(kind.split(',') if isinstance(kind, str) else kind).filter()

In [ ]:
#| export
def mk_encoder(model:str=None,      # model2vec/HF id; None -> the retrieval default
               dims:int=256,        # dims for the hashing fallback only
               offline:bool=False,  # skip the download attempt entirely
               dtype=np.float16,    # stored width; litesearch's default everywhere
) -> AttrDict:
    """The best encoder available as `AttrDict(model, doc, query, dtype, dims, method, note)`.

    Degrades to litesearch's `hash_embed` rather than failing, and always says which answered: the
    gap between the two is the gap between a vault that answers questions and one that can only
    keyword-match. `model` is the embedder object itself, so kosha can share it (see `code`).

    Both encoders are cast to `dtype` because litesearch's own default is float16 and not every
    entry point takes a `dtype=`: `Database.context` does not thread one down to its section search,
    so a float32 store would be read back as float16 there — the vectors survive, the *distances*
    do not, and `context()` silently degrades to keyword ranking. Half precision costs nothing at
    these magnitudes; a mismatched width costs the whole semantic leg."""
    if not offline:
        try:
            m = static_embedder(model or DFLT_ENC)
            v = m.encode(['probe'])
            cast = lambda f: lambda xs: np.asarray(f(xs), dtype=dtype)
            return AttrDict(model=m, doc=cast(doc_encoder(m)), query=cast(query_encoder(m)), dtype=dtype,
                            dims=int(v.shape[-1]), method='model2vec',
                            note=f'{model or DFLT_ENC} ({v.shape[-1]}d, {np.dtype(dtype)})')
        except Exception as e:
            warnings.warn(f'could not load {model or DFLT_ENC} ({type(e).__name__}: {str(e)[:100]}); '
                          f'falling back to hash_embed — retrieval will be lexical, not semantic')
    f = partial(hash_embed, ndim=dims, dtype=dtype)
    return AttrDict(model=None, doc=f, query=f, dtype=dtype, dims=dims, method='hash',
                    note=f'char-n-gram hashing ({dims}d) — lexical only; pass encoder= or restore '
                         f'network access for real semantics')

In [ ]:
#| export
class Vault:
    """Everything you have read, in one SQLite file, searchable as one corpus.

    Web pages, PDFs, papers, transcripts, local files, code and your own notes land in the same
    litesearch store under different `kind`s, which is the point: one query crosses all of them and
    `context()` hands back sections rather than fragments. Acquisition lives in
    `vishalakshi.acquire`, answering in `.ask`, code in `.code` — all optional, since the vault
    itself needs neither a network nor an LLM."""

    def __init__(self,
                 path:str=None,       # vault file; None -> ~/.vishalakshi/vault.db
                 encoder=None,        # an mk_encoder() AttrDict, a model id, or None
                 store:str='store',   # chunk store name
                 offline:bool=False,  # never attempt a model download
                 dims:int=256):       # dims for the hashing fallback
        self.path = str(ifnone(path, Path.home()/'.vishalakshi'/'vault.db'))
        self.store = store
        self.enc = encoder if isinstance(encoder, AttrDict) else mk_encoder(encoder, dims=dims, offline=offline)
        self.dtype = self.enc.dtype
        self.db = database(self.path)
        self.g = self.db.get_tree(store, dtype=self.dtype, ndim=self.enc.dims)

    def qv(self, q:str) -> bytes:
        'Query-side embedding of `q`, as the bytes every litesearch search call wants.'
        return np.asarray(self.enc.query([q])[0], dtype=self.dtype).tobytes()

    def _where(self, kind) -> str:
        'A chunk-store `WHERE` for a kind filter — pushed into the search, not applied after it.'
        return None if not kinds(kind) else f'doc_id IN (SELECT id FROM {self.g.prefix}docs WHERE {_kw(kind)})'

    def __repr__(self):
        s = self.stats()
        return (f"Vault({self.path!r}: {s['docs']} docs, {s['chunks']} chunks, "
                f"{s['entities']} entities, encoder={self.enc.method})")

def _kw(kind) -> str: return 'kind IN (%s)' % ','.join(map(repr, kinds(kind)))

In [ ]:
#| export
@patch
def add(self:Vault,
        pages,                # markdown/text, or [(page_no, text)]
        title:str,            # document title
        source:str=None,      # url or path; defaults to the title. Identity is hashed over it
        kind:str='file',      # one of KINDS — the facet you filter and report on
        meta:dict=None,       # provenance: the query that found it, when, which tier fetched it
        force:bool=False,     # re-ingest a source already present
        **kw                  # forwarded to litesearch add_doc (chunker, summarize, with_heading)
) -> dict:
    """Ingest one document into the vault: tree, chunks, embeddings, ANN index.

    Identity is content-addressed over `source|title`, so re-adding the same page is a no-op rather
    than a duplicate — which is what makes it safe to re-run a search whose results overlap what
    you already have."""
    return self.db.add_doc(pages, title, source=source, kind=kind, store=self.store,
                           emb_fn=self.enc.doc, meta=meta, force=force, **kw)

@patch
def assets(self:Vault, name:str=None) -> Path:
    'Where extracted assets (PDF images) go: beside the vault file, never the working directory.'
    d = Path(self.path).parent/'assets'
    d.mkdir(parents=True, exist_ok=True)
    return d/name if name else d

@patch
def add_file(self:Vault, path:str, title:str=None, kind:str=None, **kw) -> dict:
    """Ingest one local file: PDFs page by page, everything else through litesearch's parsers.

    PDFs are parsed here rather than through `litesearch.add_file` for one reason: pdf-oxide writes
    extracted images relative to its `out_path`, which defaults to `./pdfs/`, so ingesting a paper
    would silently litter whatever directory you happened to be in. They go next to the vault."""
    p = Path(path)
    if p.suffix.lower() == '.pdf':
        return self.add(list(enumerate(pdf_parse(str(p), out_path=self.assets(p.stem)))),
                        title or p.stem.replace('_', ' ').replace('-', ' '),
                        source=str(p), kind=kind or 'pdf', **kw)
    r = self.db.add_file(p, title=title, store=self.store, emb_fn=self.enc.doc, **kw)
    if kind and r.get('doc_id') and not r.get('skipped'): self.g.docs.update(dict(id=r['doc_id'], kind=kind))
    return r

@patch
def add_dir(self:Vault, dir:str, types:str=DOC_EXTS, kind:str=None, **kw) -> L:
    'Ingest every document under a directory. `dir2files` skips dotfiles, tests, build and dist.'
    return dir2files(dir, types=types).map(self.add_file, kind=kind, **kw)

@patch
def note(self:Vault,
         text:str,            # what you want to remember
         title:str=None,      # defaults to the first line
         tags:list=None,      # free-form tags, kept in the doc's meta
) -> dict:
    """Write a note into the vault so it is searched alongside the corpus.

    Notes are ordinary documents with `kind='note'`, which is deliberate: the graph, the clusters
    and `context()` all see them for free, so what you concluded about a corpus comes back next to
    the evidence you concluded it from."""
    ttl = title or (text.strip().splitlines() or ['note'])[0].lstrip('# ')[:80]
    return self.add(text.strip(), ttl, source=f'note:{uuid.uuid4().hex[:12]}', kind='note',
                    meta=dict(tags=list(tags or [])))

In [ ]:
#| export
@patch
def find(self:Vault,
         q:str,              # query
         limit:int=10,       # hits to return
         kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
         **kw                # forwarded to litesearch doc_search
) -> list:
    'Chunk-level hybrid search (FTS5 + vectors, RRF-fused), each hit carrying its breadcrumb.'
    hits = self.db.doc_search(q, self.qv(q), limit=limit, store=self.store, dtype=self.dtype,
                              where=self._where(kind), **kw)
    for h in hits: h['breadcrumb'] = tidy_bc(h.get('breadcrumb'))
    return hits

@patch
def sections(self:Vault, q:str, limit:int=5, kind:str=None, per:int=3, **kw) -> list:
    'Ranked *sections* rather than chunks — the unit worth reading, each with a `read` handle.'
    secs = self.db.sections(q, self.qv(q), limit=limit, per=per, store=self.store, dtype=self.dtype,
                            where=self._where(kind), **kw)
    for s in secs: s['breadcrumb'] = tidy_bc(s.get('breadcrumb'))
    return secs

@patch
def context(self:Vault,
            q:str,              # the question
            sections:int=6,     # operative sections returned
            related:int=8,      # related sections reached by graph + vector
            kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
            max_read:int=6000,  # chars of assembled text per section
            **kw                # forwarded to litesearch context
) -> AttrDict:
    """The retrieval an LLM should be handed: whole sections plus what they connect to.

    Operative sections carry `text, breadcrumb, pages, filename` and their tree neighbourhood;
    `related` holds sections reached by the entity graph (`via='graph'`) and by embedding
    similarity (`via='vector'`). `kind` filters *after* retrieval here, because litesearch's
    `context` does not thread a `where` down to both legs — the over-fetch covers the common case,
    but a filter matching very little of a large vault can still come back short."""
    keep = None if not kinds(kind) else {r['id'] for r in self.g.docs(where=_kw(kind), select='id')}
    ctx = self.db.context(q, self.qv(q), store=self.store, related=related, max_read=max_read,
                          sections=sections*3 if keep else sections, **kw)
    if keep is not None:
        ctx.results = ctx.results.filter(lambda r: r.doc_id in keep)[:sections]
        ctx.related = ctx.related.filter(lambda r: r.doc_id in keep)[:related]
    for r in (*ctx.results, *ctx.related): r.breadcrumb = tidy_bc(r.breadcrumb)
    ctx.encoder = self.enc.note
    return ctx

@patch
def related(self:Vault, node_id:str, limit:int=8) -> L:
    """Sections nearest an existing one — "what else in the vault reads like this".

    Reuses the vectors usearch already holds, so nothing is re-embedded."""
    out = {}
    for r in self.g.store(where=f'node_id={node_id!r}', select='rowid as rowid'):
        for n in self.g.store.ann_neighbors(r['rowid'], limit=limit*3, dtype=self.dtype,
                                            columns=['content', 'node_id', 'doc_id']):
            nid = n.get('node_id')
            if nid and nid != node_id and nid not in out:
                out[nid] = dict(node_id=nid, doc_id=n.get('doc_id'), dist=n.get('_dist'),
                                breadcrumb=tidy_bc(self.db.breadcrumb(nid, self.store)),
                                snippet=(n.get('content') or '')[:300])
            if len(out) >= limit: return L(out.values())
    return L(out.values())

@patch
def read(self:Vault, node_id:str, max_chars:int=6000) -> dict:
    'Assemble a whole section back out of its chunks.'
    return self.db.read(node_id, store=self.store, max_chars=max_chars)

@patch
def toc(self:Vault, **kw) -> list:
    'The table of contents across every document in the vault.'
    return self.db.toc(store=self.store, **kw)

In [ ]:
#| export
@patch
def connect(self:Vault, resolve:bool=True, **kw) -> dict:
    '(Re)build the entity graph over everything in the vault.'
    chunks = list(self.g.store())
    if not chunks: return dict(entities=0, mentions=0, edges=0, windows=0)
    self.db.get_graph(self.store, ndim=self.enc.dims, dtype=self.dtype)
    res = build_graph(self.db, chunks, store=self.store, emb_fn=self.enc.doc, **kw)
    if resolve: res = dict(res, resolved=resolve_entities(self.db, store=self.store, dtype=self.dtype))
    return res

@patch
def map(self:Vault, min_count:int=2, **kw) -> AttrDict:
    'Cluster the corpus into labelled topics — the shape of what you have collected.'
    return self.g.store.clusters(min_count=min_count, dtype=self.dtype, columns=['content', 'doc_id'], **kw)

@patch
def sources(self:Vault, kind:str=None) -> L:
    'Every document in the vault with its provenance, newest first.'
    rows = self.g.docs(where=_kw(kind) if kinds(kind) else None, order_by='added_at desc')
    return L(rows).map(lambda d: dict(d, meta=json.loads(d['meta'] or '{}')))

@patch
def forget(self:Vault, doc_id:str):
    'Remove a document, its sections and its chunks, and rebuild the ANN index.'
    self.db.delete_doc(doc_id, store=self.store)

@patch
def stats(self:Vault) -> dict:
    'Row counts across the vault, by kind.'
    p, t = self.g.prefix, self.db.t
    return dict(docs=self.g.docs.count, nodes=self.g.nodes.count, chunks=self.g.store.count,
                entities=t[f'{p}entities'].count if f'{p}entities' in t else 0,
                by_kind={r['kind']: r['n'] for r in
                         self.db.q(f'select kind, count(*) as n from {p}docs group by kind order by n desc')},
                encoder=self.enc.method, path=self.path)

## Try it

`offline=True` skips the model download and uses litesearch's `hash_embed`, which is what you want
in CI and for a quick look: retrieval is lexical, and `stats()['encoder']` says so.

In [ ]:
v = Vault(':memory:')
v.note('Late chunking beats naive chunking because context survives the split.', tags=['retrieval'])
v.add('# Attention\n\nScaled dot-product attention weights values by query-key similarity.\n\n'
      '## Multi-head\n\nHeads attend to different subspaces in parallel.', 'Attention', kind='note')
v.stats()

{'docs': 2,
 'nodes': 5,
 'chunks': 3,
 'entities': 0,
 'by_kind': {'note': 2},
 'encoder': 'model2vec',
 'path': ':memory:'}

In [ ]:
test_eq(v.stats()['docs'], 2)
test_eq(v.stats()['encoder'], 'model2vec')
assert v.find('chunking')[0]['content']
test_eq([d['kind'] for d in v.sources()], ['note', 'note'])
test_eq(len(v.sources(kind='web')), 0)
test_eq(len(v.find('chunking', kind='web')), 0)

In [ ]:
test_eq(tidy_bc('Attention › Pages 1–1: Scaled dot-product › Multi-head'), 'Attention › Multi-head')
test_eq(tidy_bc(None), '')

In [ ]:
r = v.connect()
assert r['entities'] > 0 and r['resolved']['resolvable'] == r['entities']
test_eq(v.stats()['entities'], r['entities'])

In [ ]:
#| hide
# the store, the query vector and litesearch's own default must all agree on width, or `context`
# reads f32 bytes as f16 and ranks by keyword alone
test_eq(v.enc.dtype, np.float16)
test_eq(len(v.qv('chunking')), v.enc.dims*2)
with warnings.catch_warnings():
    warnings.simplefilter('error')          # litesearch warns on a dtype mismatch; it must not fire
    assert v.context('why does late chunking help', sections=2, related=2).results

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()